# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/janaadarwish/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1) Method choice and why

**Task:** Rank pages for review (binary label used as a proxy), judged by Precision@50.

**Models compared:**
- **Baseline (Week 4):** my hand-written score.
- **Logistic Regression:** simple and readable, so its coefficients show direction.
- **Random Forest:** handles non-linear signals and interactions with little tuning.

**Why these, not something fancier:** I only keep a more complex model if it clearly beats the simple one on the same split. Complexity alone isn't a result.

In [14]:
import duckdb, numpy as np, pandas as pd
from huggingface_hub import hf_hub_download
from google.colab import userdata

TOKEN, REPO = userdata.get("HF_TOKEN"), "FlyRank/internship-warehouse"
FACT = hf_hub_download(REPO, "fact_content_daily_performance/month=2026-03/data_0.parquet", repo_type="dataset", token=TOKEN)
DIM_CONTENT = hf_hub_download(REPO, "dim_content.parquet", repo_type="dataset", token=TOKEN)
con = duckdb.connect()

df = con.sql(f"""
WITH early AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS impressions_early,
         SUM(gsc_clicks) AS clicks_early,
         AVG(gsc_avg_position) AS avg_position_early,
         SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-10') AS imp_h1,
         SUM(gsc_impressions) FILTER (WHERE report_date >  DATE '2026-03-10') AS imp_h2
  FROM read_parquet('{FACT}')
  WHERE gsc_data_available IS TRUE AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-20'
  GROUP BY 1, 2),
late AS (
  SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_late
  FROM read_parquet('{FACT}')
  WHERE gsc_data_available IS TRUE AND report_date BETWEEN DATE '2026-03-21' AND DATE '2026-03-31'
  GROUP BY 1, 2)
SELECT e.client_hash_id, e.content_hash_id, e.impressions_early,
       e.clicks_early * 1.0 / NULLIF(e.impressions_early, 0) AS ctr_early,
       e.avg_position_early,
       (COALESCE(e.imp_h2, 0) + 1.0) / (COALESCE(e.imp_h1, 0) + 1.0) AS impr_trend_early,
       DATE '2026-03-20' - c.content_updated_date AS days_since_update,
       c.search_volume,
       (COALESCE(l.clicks_late, 0) / 11.0) < 0.8 * (e.clicks_early / 20.0) AS label
FROM early e
JOIN read_parquet('{DIM_CONTENT}') c USING (client_hash_id, content_hash_id)
LEFT JOIN late l USING (client_hash_id, content_hash_id)
WHERE e.clicks_early >= 5 AND c.content_updated_date <= DATE '2026-03-20'
""").df()

df["label"] = df["label"].astype(int)
df["days_since_update"] = df["days_since_update"].astype(float)
df["search_volume"] = df["search_volume"].astype(float)
print(df.shape, "| share label=1:", round(df["label"].mean(), 3))
print("missing share:\n", df[["impr_trend_early", "search_volume"]].isna().mean().round(3))

(2526, 9) | share label=1: 0.59
missing share:
 impr_trend_early    0.000
search_volume       0.016
dtype: float64


## 2) Split design

I used a **grouped split by client** (`client_hash_id`), with no client in both train and test (the overlap check prints 0), so the model is tested on clients it hasn't seen. A random row split would let the model learn a client's habits and look better than it is.

Development uses month=2026-03 only. June 2026 stays sealed and is not touched in this notebook.

In [15]:
from sklearn.model_selection import GroupShuffleSplit

df["search_volume"] = df["search_volume"].fillna(df["search_volume"].median())
df["impressions_log"] = np.log1p(df["impressions_early"])
df["search_volume_log"] = np.log1p(df["search_volume"])
df["impr_trend_log"] = np.log(df["impr_trend_early"].clip(0.05, 20))

# Week 4 baseline rule, recomputed here (uses only early-window signals, never the label)
def pct(s): return s.rank(pct=True)
df["score"] = 0.8 * (1 - pct(df["impressions_early"])) + 0.2 * pct(df["days_since_update"])

FEATS = ["impressions_log", "ctr_early", "avg_position_early", "days_since_update", "search_volume_log", "impr_trend_log"]
LABEL, GROUP = "label", "client_hash_id"

d = df.dropna(subset=FEATS + [LABEL]).reset_index(drop=True)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
tr_idx, te_idx = next(gss.split(d, groups=d[GROUP]))
train, test = d.iloc[tr_idx], d.iloc[te_idx]

print("train:", len(train), "test:", len(test))
print("clients in train / test:", train[GROUP].nunique(), "/", test[GROUP].nunique())
print("groups overlap:", len(set(train[GROUP]) & set(test[GROUP])))   # should print 0

train: 1126 test: 1400
clients in train / test: 15 / 7
groups overlap: 0


Errors and interpretation

- The most important features were **impressions trend within the early window** (`impr_trend_log`, 0.066) and **early impressions** (`impressions_log`, 0.062), measured by permutation importance on the test split. The other four features add very little (0.012 or less). `search_volume_log` is zero for most pages, so it carries little information.
- All 8 false positives in the top 50 come from a single client, and every one of them has a falling impressions trend (log ratio between -0.5 and -2.1). The model expects pages that are already declining to keep declining, and these pages did not drop below 80% of their early click rate.
- This suggests the model leans on momentum and misses what separates pages that keep declining from pages that level off. A feature such as position change within the early window might help.

**Careful words:** These results are observed on a sample and are directional. Feature importance describes the model, not causes in search.

**Important limit:** The test split is dominated by one client (1,353 of about 1,400 test rows, 96%). The Precision@50, AUC and error analysis therefore mostly describe that client and may not generalise to others. Results are directional.

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

def precision_at_k(y_true, score, k=50):
    order = np.argsort(-np.asarray(score))[:k]
    return float(np.asarray(y_true)[order].mean())

Xtr, ytr, Xte, yte = train[FEATS], train[LABEL], test[FEATS], test[LABEL]
base_rate = round(float(yte.mean()), 3)

rows = [{"Method": "Baseline rule (Week 4)",
         "Precision@50": round(precision_at_k(yte, test["score"], 50), 3),
         "AUC": round(roc_auc_score(yte, test["score"]), 3)}]

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1),
}
fitted, scores = {}, {}
for name, m in models.items():
    m.fit(Xtr, ytr)
    s = m.predict_proba(Xte)[:, 1]
    fitted[name], scores[name] = m, s
    rows.append({"Method": name, "Precision@50": round(precision_at_k(yte, s, 50), 3), "AUC": round(roc_auc_score(yte, s), 3)})

print("Base rate in the test split:", base_rate)
comparison = pd.DataFrame(rows)
comparison

Base rate in the test split: 0.536


,Method,Precision@50,AUC
0,Baseline rule (Week 4),0.82,0.625
1,Logistic Regression,0.84,0.637
2,Random Forest,0.72,0.602


In [17]:
from sklearn.inspection import permutation_importance

best = max(scores, key=lambda n: roc_auc_score(yte, scores[n]))
print("Best by AUC:", best)
r = permutation_importance(fitted[best], Xte, yte, n_repeats=5, random_state=0, scoring="roc_auc")
pd.Series(r.importances_mean, index=FEATS).sort_values(ascending=False).round(4)

Best by AUC: Logistic Regression


,0
impr_trend_log,0.0563
impressions_log,0.0557
search_volume_log,0.0121
ctr_early,0.0026
avg_position_early,0.0026
days_since_update,0.0002


In [18]:
test = test.copy()
test["model_score"] = scores[best]
top = test.sort_values("model_score", ascending=False).head(50)
print("False positives in top 50:", int((top[LABEL] == 0).sum()))
top[top[LABEL] == 0][[GROUP] + FEATS + ["model_score"]].round(3).head(10)

False positives in top 50: 8


,client_hash_id,impressions_log,ctr_early,avg_position_early,days_since_update,search_volume_log,impr_trend_log,model_score
1312,client_73cda7b4e4f265ea,8.251,0.003,5.196,23.0,0.000,-1.793,0.764
657,client_73cda7b4e4f265ea,7.465,0.004,4.031,23.0,0.000,-1.247,0.747
2025,client_73cda7b4e4f265ea,8.381,0.001,4.727,23.0,3.045,-2.107,0.738
2473,client_73cda7b4e4f265ea,7.116,0.005,2.446,23.0,0.000,-0.809,0.726
633,client_73cda7b4e4f265ea,7.790,0.003,2.394,23.0,0.000,-0.961,0.723
1191,client_73cda7b4e4f265ea,6.666,0.008,2.483,23.0,0.000,-0.659,0.722
1831,client_73cda7b4e4f265ea,7.587,0.004,3.377,23.0,0.000,-0.769,0.716
2461,client_73cda7b4e4f265ea,6.853,0.006,2.038,23.0,0.000,-0.536,0.712


In [19]:
print(test[GROUP].value_counts().head(5))
print(top[GROUP].value_counts().head(5))

client_hash_id
client_73cda7b4e4f265ea    1353
client_b10cb2997d0c7c86      18
client_ff644d8251367cbb      10
client_a80fca3f171ed1de      10
client_08d2847f24cf89c1       4
Name: count, dtype: int64
client_hash_id
client_73cda7b4e4f265ea    46
client_b10cb2997d0c7c86     2
client_f623b01661d4bfe4     1
client_08d2847f24cf89c1     1
Name: count, dtype: int64


## 3) Model vs baseline (same split, same metric)

On the same grouped test split (base rate ___): Precision@50 was ___ for the baseline, ___ for Logistic Regression and ___ for Random Forest. AUC was ___, ___ and ___.

**My read:** ___ (say whether a model beats the baseline, by how much, and whether Logistic Regression gets most of the way. If a model doesn't beat the baseline, say so plainly. It's a valid result.)

**Note:** Precision@50 uses only 50 pages, so differences of a few points are within noise. I weight AUC more for the comparison.


## 5) Self-check

- [x] Compared to the Week 4 baseline on the same split and metric
- [x] Used a grouped split by client, with the overlap check printed
- [x] Explained why these methods
- [x] Reported Precision@50 and AUC
- [x] Interpreted features and errors
- [x] Didn't reward complexity alone